# Prédiction des Positions Futures de Navires (Besoin Client 3)

**Date** : 11 Juin 2025  
**Auteur** : Marine REVERDY  
**Promotion** : A3 ISEN

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [2]:

# Chargement des données
vessel_init = pd.read_csv("export_IA.csv")

# Conversion des données
vessel_clean = vessel_init.astype({
    'LAT': 'float32',
    'LON': 'float32',
    'SOG': 'float32',
    'COG': 'float32',
    'Heading': 'float32',
    'VesselType': 'int32',
    'Length': 'float32',
    'Width': 'float32',
    'Draft': 'float32',
    'VesselName': 'str',
    'Status': 'int32'
})
vessel_clean['BaseDateTime'] = pd.to_datetime(vessel_clean['BaseDateTime'])

# Colonnes à garder et nettoyage NaN
cols = ['VesselName', 'MMSI', 'BaseDateTime', 'LAT', 'LON', 'SOG', 'COG', 'Heading', 'VesselType', 'Length', 'Width', 'Draft', 'Status']
vessel_clean = vessel_clean[cols].dropna()

In [3]:

horizons = [5, 10, 15]  # horizons en minutes
speed_threshold = 1  # seuil de vitesse en noeuds

vessel_selected = []

for vessel_name, group in vessel_clean.groupby("VesselName"):
    group = group.sort_values("BaseDateTime")
    group = group[group['SOG'] > speed_threshold]
    group = group[group['Status'] != 5]  # Exclure les statuts 5 (arrêté)
    if group.empty:
        continue
    

    # Intervalle moyen en minutes
    time_diffs = group['BaseDateTime'].diff().dropna()
    interval = time_diffs.dt.total_seconds().mean() / 60
    if interval == 0 or np.isnan(interval):
        continue

    shifts = {h: int(round(h / interval)) for h in horizons}

    # Création des colonnes LAT+horizon et LON+horizon (positions futures)
    for h, shift in shifts.items():
        group[f'LAT+{h}'] = group['LAT'].shift(-shift)
        group[f'LON+{h}'] = group['LON'].shift(-shift)

    vessel_selected.append(group)


# Concaténer tous les groupes
vessel_selected = pd.concat(vessel_selected, ignore_index=True)
vessel_selected.dropna(inplace=True)  # supprimer lignes avec NaN (fin traj)

In [4]:
# Sélection des features et targets
features_targets = [ 'LAT', 'LON', 'SOG', 'COG', 'Heading', 'VesselType', 'Length', 'Width', 'Draft']
X = vessel_selected[features_targets]


# Targets : LAT et LON à prédire pour chaque horizon
y_lat = vessel_selected[[f'LAT+{h}' for h in horizons]]
y_lon = vessel_selected[[f'LON+{h}' for h in horizons]]

# Séparer les index pour les traçages futurs
indices = vessel_selected.index.values 

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [5]:
# Split avec les index en parallèle
X_train, X_temp, y_lat_train, y_lat_temp, y_lon_train, y_lon_temp, idx_train, idx_temp = train_test_split(
    X_scaled, y_lat, y_lon, indices, test_size=0.3, random_state=42)

X_val, X_test, y_lat_val, y_lat_test, y_lon_val, y_lon_test, idx_val, idx_test = train_test_split(
    X_temp, y_lat_temp, y_lon_temp, idx_temp, test_size=0.5, random_state=42)

# Nettoyage des NaNs avec index
mask_train = ~np.isnan(X_train).any(axis=1) & ~np.isnan(y_lat_train).any(axis=1) & ~np.isnan(y_lon_train).any(axis=1)
X_train_clean = X_train[mask_train]
y_lat_train_clean = y_lat_train[mask_train]
y_lon_train_clean = y_lon_train[mask_train]

mask_val = ~np.isnan(X_val).any(axis=1) & ~np.isnan(y_lat_val).any(axis=1) & ~np.isnan(y_lon_val).any(axis=1)
X_val_clean = X_val[mask_val]
y_lat_val_clean = y_lat_val[mask_val]
y_lon_val_clean = y_lon_val[mask_val]
idx_val_clean = idx_val[mask_val]

mask_test = ~np.isnan(X_test).any(axis=1) & ~np.isnan(y_lat_test).any(axis=1) & ~np.isnan(y_lon_test).any(axis=1)
X_test_clean = X_test[mask_test]
y_lat_test_clean = y_lat_test[mask_test]
y_lon_test_clean = y_lon_test[mask_test]
idx_test_clean = idx_test[mask_test]

In [ ]:
# Entraînement et évaluation
for horizon in horizons:
    print(f"\nEntraînement pour l'horizon {horizon} minutes...")
    
    # Entraînement du modèle
    model_lat = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42).fit(X_train, y_lat_train[f'LAT+{horizon}'])
    model_lon = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42).fit(X_train, y_lon_train[f'LON+{horizon}'])

    joblib.dump(model_lat, f"model_lat_{horizon}.pkl")
    joblib.dump(model_lon, f"model_lon_{horizon}.pkl")

    pred_lat_val = model_lat.predict(X_val_clean)
    pred_lon_val = model_lon.predict(X_val_clean)

    r2_lat = r2_score(y_lat_val_clean[f'LAT+{horizon}'], pred_lat_val)
    r2_lon = r2_score(y_lon_val_clean[f'LON+{horizon}'], pred_lon_val)
    rmse_lat = np.sqrt(mean_squared_error(y_lat_val_clean[f'LAT+{horizon}'], pred_lat_val))
    rmse_lon = np.sqrt(mean_squared_error(y_lon_val_clean[f'LON+{horizon}'], pred_lon_val))

    print(f"\nEvaluation pour l'horizon {horizon} minutes :")

    print(f"[VALID] Horizon {horizon} minutes : R2 LAT = {r2_lat:.4f}, R2 LON = {r2_lon:.4f}")
    print(f"[VALID] RMSE LAT = {rmse_lat:.5f} | RMSE LON = {rmse_lon:.5f}")


    # --- TEST FINAL ---
    pred_lat_test = model_lat.predict(X_test_clean)
    pred_lon_test = model_lon.predict(X_test_clean)
    r2_lat_test = r2_score(y_lat_test_clean[f'LAT+{horizon}'], pred_lat_test)
    r2_lon_test = r2_score(y_lon_test_clean[f'LON+{horizon}'], pred_lon_test)
    rmse_lat_test = np.sqrt(mean_squared_error(y_lat_test_clean[f'LAT+{horizon}'], pred_lat_test))
    rmse_lon_test = np.sqrt(mean_squared_error(y_lon_test_clean[f'LON+{horizon}'], pred_lon_test))

    print(f"\nTest pour l'horizon {horizon} minutes :")

    print(f"[TEST] Horizon {horizon} min : R2 LAT = {r2_lat_test:.4f}, R2 LON = {r2_lon_test:.4f}")
    print(f"[TEST] RMSE LAT = {rmse_lat_test:.5f} | RMSE LON = {rmse_lon_test:.5f}")
    


Entraînement pour l'horizon 5 minutes...
